In [30]:
# ============================================================
# STEP 6 - CELL 1
# SETUP
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix

import json
import pickle
import gc
import warnings
import time

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/step6")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RANDOM_STATE = 42

np.random.seed(
    RANDOM_STATE
)

print("=" * 70)
print("STEP 6 - COLLABORATIVE FILTERING")
print("=" * 70)

print("Input :", INPUT_DIR)
print("Output:", OUTPUT_DIR)

STEP 6 - COLLABORATIVE FILTERING
Input : /kaggle/input
Output: /kaggle/working/step6


In [31]:
# ============================================================
# STEP 6 - CELL 2
# LOAD CF DATASET
# ============================================================

def find_file(filename):

    matches = list(
        INPUT_DIR.rglob(filename)
    )

    if not matches:

        raise FileNotFoundError(
            f"{filename} not found"
        )

    return matches[0]


cf_path = find_file(
    "cf_interactions.csv"
)

print(
    "CF file:",
    cf_path
)


cf = pd.read_csv(
    cf_path,
    low_memory=False
)


print("\n" + "=" * 70)
print("CF DATASET")
print("=" * 70)

print(
    "Interactions:",
    f"{len(cf):,}"
)

print(
    "Users:",
    f"{cf['user_id'].nunique():,}"
)

print(
    "Products:",
    f"{cf['product_id'].nunique():,}"
)

print("\nColumns:")
print(
    cf.columns.tolist()
)

display(
    cf.head()
)

CF file: /kaggle/input/datasets/trnquangtriu/6dataset/cf_interactions.csv

CF DATASET
Interactions: 259,427
Users: 26,903
Products: 15,626

Columns:
['user_id', 'product_id', 'rating', 'timestamp', 'source']


,user_id,product_id,rating,timestamp,source
0,amazon_AHF7C34PIPFVLHILFJLYNALWI2ZQ,amazon_B00WUF45UO,4.0,963431699000,amazon
1,amazon_AHF7C34PIPFVLHILFJLYNALWI2ZQ,amazon_B07KQWX947,4.0,963432107000,amazon
2,amazon_AH7ZES2DTJZAMNI333OWGJBRWJTQ,amazon_B00004VUGJ,1.0,980357976000,amazon
3,amazon_AHWU6P7U44MBK4M4QV2TOTTKHIFA,amazon_B00004SB92,5.0,998969441000,amazon
4,amazon_AHMBV2DCWTL2DXHJUWTRQQVM4YBQ,amazon_B01AHWH83M,5.0,1010096162000,amazon


# CF DATA AUDIT


In [32]:
# ============================================================
# STEP 6 - CELL 3
# CF DATA AUDIT
# ============================================================

print("=" * 70)
print("CF DATA AUDIT")
print("=" * 70)


required = [
    "user_id",
    "product_id",
    "rating",
    "timestamp"
]


for col in required:

    assert col in cf.columns, (
        f"Missing column: {col}"
    )


print(
    "Missing user:",
    cf["user_id"].isna().sum()
)

print(
    "Missing product:",
    cf["product_id"].isna().sum()
)

print(
    "Missing rating:",
    cf["rating"].isna().sum()
)

print(
    "Missing timestamp:",
    cf["timestamp"].isna().sum()
)


duplicate_pairs = (
    cf.duplicated(
        [
            "user_id",
            "product_id"
        ]
    ).sum()
)


print(
    "\nDuplicate user-product:",
    duplicate_pairs
)


rating_numeric = pd.to_numeric(
    cf["rating"],
    errors="coerce"
)


print(
    "Rating range:",
    rating_numeric.min(),
    "->",
    rating_numeric.max()
)


user_counts = (
    cf["user_id"]
    .value_counts()
)

product_counts = (
    cf["product_id"]
    .value_counts()
)


print(
    "\nMin interactions/user:",
    user_counts.min()
)

print(
    "Min interactions/product:",
    product_counts.min()
)


assert duplicate_pairs == 0
assert user_counts.min() >= 5
assert product_counts.min() >= 5

print(
    "\n✅ CF dataset passed audit"
)

CF DATA AUDIT
Missing user: 0
Missing product: 0
Missing rating: 0
Missing timestamp: 0

Duplicate user-product: 0
Rating range: 1.0 -> 5.0

Min interactions/user: 5
Min interactions/product: 5

✅ CF dataset passed audit


# Chuẩn hóa Timestamp

In [33]:
# ============================================================
# STEP 6 - CELL 4
# NORMALIZE TIMESTAMP
# ============================================================

cf = cf.copy()

cf["timestamp"] = pd.to_numeric(
    cf["timestamp"],
    errors="coerce"
)


median_timestamp = (
    cf["timestamp"]
    .median()
)


if median_timestamp > 1e11:

    TIMESTAMP_UNIT = "ms"

else:

    TIMESTAMP_UNIT = "s"


cf["datetime"] = pd.to_datetime(
    cf["timestamp"],
    unit=TIMESTAMP_UNIT,
    errors="coerce"
)


print(
    "Detected unit:",
    TIMESTAMP_UNIT
)

print(
    "Earliest:",
    cf["datetime"].min()
)

print(
    "Latest  :",
    cf["datetime"].max()
)

print(
    "Missing datetime:",
    cf["datetime"].isna().sum()
)


assert (
    cf["datetime"].notna().all()
)

print(
    "\n✅ Timestamp ready"
)

Detected unit: ms
Earliest: 2000-07-12 19:54:59
Latest  : 2023-03-20 01:34:02.696000
Missing datetime: 0

✅ Timestamp ready


# TEMPORAL LEAVE-TWO-OUT SPLIT


In [34]:
# ============================================================
# STEP 6 - CELL 5
# TEMPORAL LEAVE-TWO-OUT SPLIT
# ============================================================

print("=" * 70)
print("TEMPORAL TRAIN / VALIDATION / TEST SPLIT")
print("=" * 70)


cf_sorted = (
    cf
    .sort_values(
        [
            "user_id",
            "timestamp",
            "product_id"
        ]
    )
    .reset_index(drop=True)
)


cf_sorted[
    "reverse_rank"
] = (
    cf_sorted
    .groupby(
        "user_id"
    )
    .cumcount(
        ascending=False
    )
)


test = (
    cf_sorted[
        cf_sorted[
            "reverse_rank"
        ] == 0
    ]
    .drop(
        columns="reverse_rank"
    )
    .reset_index(drop=True)
)


validation = (
    cf_sorted[
        cf_sorted[
            "reverse_rank"
        ] == 1
    ]
    .drop(
        columns="reverse_rank"
    )
    .reset_index(drop=True)
)


train = (
    cf_sorted[
        cf_sorted[
            "reverse_rank"
        ] >= 2
    ]
    .drop(
        columns="reverse_rank"
    )
    .reset_index(drop=True)
)


print(
    "Train      :",
    f"{len(train):,}"
)

print(
    "Validation :",
    f"{len(validation):,}"
)

print(
    "Test       :",
    f"{len(test):,}"
)


print(
    "\nTrain users:",
    f"{train['user_id'].nunique():,}"
)

print(
    "Valid users:",
    f"{validation['user_id'].nunique():,}"
)

print(
    "Test users :",
    f"{test['user_id'].nunique():,}"
)


print(
    "\nMin train interactions/user:",
    train[
        "user_id"
    ].value_counts().min()
)

TEMPORAL TRAIN / VALIDATION / TEST SPLIT
Train      : 205,621
Validation : 26,903
Test       : 26,903

Train users: 26,903
Valid users: 26,903
Test users : 26,903

Min train interactions/user: 3


# TEMPORAL LEAKAGE AUDIT


In [35]:
# ============================================================
# STEP 6 - CELL 6
# TEMPORAL LEAKAGE AUDIT
# ============================================================

print("=" * 70)
print("TEMPORAL LEAKAGE AUDIT")
print("=" * 70)


train_max = (
    train
    .groupby("user_id")[
        "timestamp"
    ]
    .max()
)


valid_time = (
    validation
    .set_index("user_id")[
        "timestamp"
    ]
)


test_time = (
    test
    .set_index("user_id")[
        "timestamp"
    ]
)


users = (
    sorted(
        set(train_max.index)
        &
        set(valid_time.index)
        &
        set(test_time.index)
    )
)


train_max = (
    train_max.loc[users]
)

valid_time = (
    valid_time.loc[users]
)

test_time = (
    test_time.loc[users]
)


train_before_valid = (
    train_max
    <= valid_time
).all()


valid_before_test = (
    valid_time
    <= test_time
).all()


print(
    "Train <= Validation:",
    train_before_valid
)

print(
    "Validation <= Test:",
    valid_before_test
)

print(
    "Users checked:",
    f"{len(users):,}"
)


assert train_before_valid
assert valid_before_test

print(
    "\n✅ No future interaction leakage"
)

TEMPORAL LEAKAGE AUDIT
Train <= Validation: True
Validation <= Test: True
Users checked: 26,903

✅ No future interaction leakage


# INSTALL / IMPORT IMPLICIT


In [36]:
# ============================================================
# STEP 6 - CELL 8
# INSTALL / IMPORT IMPLICIT
# ============================================================

try:

    import implicit

    from implicit.bpr import (
        BayesianPersonalizedRanking
    )

    print(
        "implicit:",
        implicit.__version__
    )

except ImportError:

    !pip install -q implicit

    import implicit

    from implicit.bpr import (
        BayesianPersonalizedRanking
    )

    print(
        "implicit:",
        implicit.__version__
    )


print(
    "✅ implicit ready"
)

implicit: 0.7.3
✅ implicit ready


# BUILD TRAIN USER / ITEM MAPPINGS


In [37]:
# ============================================================
# STEP 6 - CELL 9
# BUILD TRAIN USER / ITEM MAPPINGS
# ============================================================

train_users = (
    train[
        "user_id"
    ]
    .drop_duplicates()
    .tolist()
)


train_items_list = (
    train[
        "product_id"
    ]
    .drop_duplicates()
    .tolist()
)


user_to_idx = {

    user_id: idx

    for idx, user_id
    in enumerate(
        train_users
    )
}


item_to_idx = {

    product_id: idx

    for idx, product_id
    in enumerate(
        train_items_list
    )
}


idx_to_user = np.array(
    train_users,
    dtype=object
)

idx_to_item = np.array(
    train_items_list,
    dtype=object
)


print(
    "Mapped users:",
    f"{len(user_to_idx):,}"
)

print(
    "Mapped items:",
    f"{len(item_to_idx):,}"
)

Mapped users: 26,903
Mapped items: 15,546


# BUILD USER-ITEM MATRIX


In [38]:
# ============================================================
# STEP 6 - CELL 10
# BUILD USER-ITEM MATRIX
# ============================================================

train_matrix_df = (
    train[
        [
            "user_id",
            "product_id",
            "rating"
        ]
    ]
    .copy()
)


train_matrix_df[
    "user_idx"
] = train_matrix_df[
    "user_id"
].map(
    user_to_idx
)


train_matrix_df[
    "item_idx"
] = train_matrix_df[
    "product_id"
].map(
    item_to_idx
)


ratings = pd.to_numeric(
    train_matrix_df[
        "rating"
    ],
    errors="coerce"
).fillna(1.0)


weights = (
    ratings
    .clip(
        1,
        5
    )
    / 5.0
)


user_item_matrix = csr_matrix(

    (
        weights.astype(
            np.float32
        ),

        (
            train_matrix_df[
                "user_idx"
            ].values,

            train_matrix_df[
                "item_idx"
            ].values
        )
    ),

    shape=(
        len(user_to_idx),
        len(item_to_idx)
    ),

    dtype=np.float32
)


print("=" * 70)
print("USER-ITEM MATRIX")
print("=" * 70)

print(
    "Shape:",
    user_item_matrix.shape
)

print(
    "Non-zero:",
    f"{user_item_matrix.nnz:,}"
)


density = (
    user_item_matrix.nnz
    /
    (
        user_item_matrix.shape[0]
        *
        user_item_matrix.shape[1]
    )
)


print(
    "Density :",
    f"{density:.8f}"
)

print(
    "Sparsity:",
    f"{(1-density)*100:.6f}%"
)

USER-ITEM MATRIX
Shape: (26903, 15546)
Non-zero: 205,621
Density : 0.00049164
Sparsity: 99.950836%


# POPULARITY BASELINE


In [39]:
# ============================================================
# STEP 6 - CELL 11
# POPULARITY BASELINE
# ============================================================

print("=" * 70)
print("POPULARITY BASELINE")
print("=" * 70)


item_popularity = (
    train
    .groupby(
        "product_id"
    )
    .agg(
        interaction_count=(
            "user_id",
            "size"
        ),
        avg_rating=(
            "rating",
            "mean"
        )
    )
    .reset_index()
)


item_popularity[
    "popularity_score"
] = (
    item_popularity[
        "interaction_count"
    ]
)


item_popularity = (
    item_popularity
    .sort_values(
        [
            "popularity_score",
            "avg_rating"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)


popular_items = (
    item_popularity[
        "product_id"
    ].tolist()
)


display(
    item_popularity.head(10)
)

POPULARITY BASELINE


,product_id,interaction_count,avg_rating,popularity_score
0,amazon_B01K8B8YA8,1389,4.395968,1389
1,amazon_B07S764D9V,938,4.302772,938
2,amazon_B011BRUOMO,805,4.696894,805
3,amazon_B0BGNG1294,623,4.749599,623
4,amazon_B0BW4PFM58,527,4.377609,527
5,amazon_B00X5C22SS,469,4.140725,469
6,amazon_B07P9V8GSH,453,4.666667,453
7,amazon_B07KTYJ769,447,4.561521,447
8,amazon_B01G8JO5F2,418,4.148325,418
9,amazon_B00L0YLRUW,401,3.970075,401


# RANKING METRICS


In [40]:
# ============================================================
# STEP 6 - CELL 12
# RANKING METRICS
# ============================================================

def evaluate_ranked_items(
    ranked_items,
    target_item,
    k
):

    ranked_items = list(
        ranked_items[:k]
    )


    if target_item not in ranked_items:

        return {
            "hit": 0.0,
            "recall": 0.0,
            "ndcg": 0.0,
            "mrr": 0.0
        }


    rank = (
        ranked_items.index(
            target_item
        )
        + 1
    )


    return {

        "hit":
            1.0,

        "recall":
            1.0,

        "ndcg":
            1.0
            /
            np.log2(
                rank + 1
            ),

        "mrr":
            1.0
            /
            rank
    }

# POPULARITY RECOMMENDER


In [41]:
# ============================================================
# STEP 6 - CELL 13
# POPULARITY RECOMMENDER
# ============================================================

user_train_items = (
    train
    .groupby(
        "user_id"
    )[
        "product_id"
    ]
    .apply(set)
    .to_dict()
)


def recommend_popularity(
    user_id,
    k=10
):

    seen = user_train_items.get(
        user_id,
        set()
    )


    recommendations = []

    for item in popular_items:

        if item in seen:
            continue

        recommendations.append(
            item
        )

        if (
            len(recommendations)
            >= k
        ):
            break


    return recommendations

In [42]:
# ============================================================
# STEP 6 - FIX POPULARITY EVALUATION
# ============================================================

def evaluate_popularity(
    holdout_df,
    ks=(5, 10, 20)
):

    holdout_df = holdout_df.copy()

    # Automatically identify CF-evaluable target items
    holdout_df["known_item"] = (
        holdout_df["product_id"]
        .isin(item_to_idx)
    )

    evaluable = (
        holdout_df[
            holdout_df["known_item"]
        ]
        .copy()
    )

    max_k = max(ks)

    metric_store = {
        k: {
            "hit": [],
            "recall": [],
            "ndcg": [],
            "mrr": []
        }
        for k in ks
    }

    for row in evaluable.itertuples():

        ranked = recommend_popularity(
            row.user_id,
            k=max_k
        )

        for k in ks:

            metrics = evaluate_ranked_items(
                ranked,
                row.product_id,
                k
            )

            for metric, value in metrics.items():

                metric_store[k][metric].append(
                    value
                )

    rows = []

    for k in ks:

        rows.append({
            "model": "Popularity",
            "K": k,
            "users": len(evaluable),

            "HitRate": np.mean(
                metric_store[k]["hit"]
            ),

            "Recall": np.mean(
                metric_store[k]["recall"]
            ),

            "NDCG": np.mean(
                metric_store[k]["ndcg"]
            ),

            "MRR": np.mean(
                metric_store[k]["mrr"]
            )
        })

    return pd.DataFrame(rows)


print("✅ Robust evaluate_popularity() ready")

✅ Robust evaluate_popularity() ready


# Train BPR Configurations

In [43]:
# ============================================================
# STEP 6 - CELL 15
# BPR CONFIGURATIONS
# ============================================================

BPR_CONFIGS = [

    {
        "name":
            "BPR_F32_R01",

        "factors":
            32,

        "regularization":
            0.01,

        "iterations":
            50,

        "learning_rate":
            0.05
    },

    {
        "name":
            "BPR_F64_R01",

        "factors":
            64,

        "regularization":
            0.01,

        "iterations":
            50,

        "learning_rate":
            0.05
    },

    {
        "name":
            "BPR_F64_R05",

        "factors":
            64,

        "regularization":
            0.05,

        "iterations":
            50,

        "learning_rate":
            0.05
    }
]


pd.DataFrame(
    BPR_CONFIGS
)

,name,factors,regularization,iterations,learning_rate
0,BPR_F32_R01,32,0.01,50,0.05
1,BPR_F64_R01,64,0.01,50,0.05
2,BPR_F64_R05,64,0.05,50,0.05


In [44]:
# ============================================================
# STEP 6 - CELL 16
# BPR RECOMMENDATION FUNCTION
# ============================================================

def recommend_bpr(
    model,
    user_id,
    k=10
):

    if user_id not in user_to_idx:

        return []


    user_idx = (
        user_to_idx[
            user_id
        ]
    )


    ids, scores = model.recommend(

        userid=user_idx,

        user_items=user_item_matrix[
            user_idx
        ],

        N=k,

        filter_already_liked_items=True
    )


    product_ids = (
        idx_to_item[
            ids
        ]
    )


    return (
        product_ids.tolist()
    )

In [45]:
# ============================================================
# STEP 6 - CELL 17
# ROBUST BPR EVALUATION
# ============================================================

def evaluate_bpr(
    model,
    holdout_df,
    ks=(5, 10, 20)
):

    holdout_df = holdout_df.copy()

    # --------------------------------------------------------
    # Automatically determine whether target item exists
    # in the CF training catalog.
    # --------------------------------------------------------

    holdout_df["known_item"] = (
        holdout_df["product_id"]
        .isin(item_to_idx)
    )

    evaluable = (
        holdout_df[
            holdout_df["known_item"]
        ]
        .copy()
    )

    print(
        f"Evaluable users: "
        f"{len(evaluable):,} / {len(holdout_df):,} "
        f"({len(evaluable)/len(holdout_df)*100:.2f}%)"
    )

    max_k = max(ks)

    metric_store = {
        k: {
            "hit": [],
            "recall": [],
            "ndcg": [],
            "mrr": []
        }
        for k in ks
    }

    for row in evaluable.itertuples():

        ranked = recommend_bpr(
            model,
            row.user_id,
            k=max_k
        )

        for k in ks:

            metrics = evaluate_ranked_items(
                ranked,
                row.product_id,
                k
            )

            for metric, value in metrics.items():

                metric_store[k][metric].append(
                    value
                )

    rows = []

    for k in ks:

        rows.append({
            "K": k,
            "users": len(evaluable),

            "HitRate": np.mean(
                metric_store[k]["hit"]
            ),

            "Recall": np.mean(
                metric_store[k]["recall"]
            ),

            "NDCG": np.mean(
                metric_store[k]["ndcg"]
            ),

            "MRR": np.mean(
                metric_store[k]["mrr"]
            )
        })

    return pd.DataFrame(rows)


print("✅ Robust evaluate_bpr() ready")

✅ Robust evaluate_bpr() ready


# Train + Validate BPR

In [46]:
# ============================================================
# STEP 6 - FIX
# REBUILD HELD-OUT ITEM COVERAGE
# ============================================================

print("=" * 70)
print("REBUILD HELD-OUT ITEM COVERAGE")
print("=" * 70)

# Products actually available in training matrix
train_item_set = set(
    train["product_id"]
    .dropna()
    .unique()
)

# Recreate safely
validation = validation.copy()
test = test.copy()

validation["known_item"] = (
    validation["product_id"]
    .isin(train_item_set)
)

test["known_item"] = (
    test["product_id"]
    .isin(train_item_set)
)

print("Train products:", f"{len(train_item_set):,}")

print("\nValidation")
print(
    "Known items :",
    f"{validation['known_item'].sum():,}",
    f"({validation['known_item'].mean()*100:.2f}%)"
)
print(
    "Cold items  :",
    f"{(~validation['known_item']).sum():,}",
    f"({(~validation['known_item']).mean()*100:.2f}%)"
)

print("\nTest")
print(
    "Known items :",
    f"{test['known_item'].sum():,}",
    f"({test['known_item'].mean()*100:.2f}%)"
)
print(
    "Cold items  :",
    f"{(~test['known_item']).sum():,}",
    f"({(~test['known_item']).mean()*100:.2f}%)"
)

assert "known_item" in validation.columns
assert "known_item" in test.columns

print("\nValidation columns:")
print(validation.columns.tolist())

print("\n✅ known_item rebuilt successfully")

REBUILD HELD-OUT ITEM COVERAGE
Train products: 15,546

Validation
Known items : 26,745 (99.41%)
Cold items  : 158 (0.59%)

Test
Known items : 26,585 (98.82%)
Cold items  : 318 (1.18%)

Validation columns:
['user_id', 'product_id', 'rating', 'timestamp', 'source', 'datetime', 'known_item']

✅ known_item rebuilt successfully


In [47]:
# ============================================================
# STEP 6 - CELL 18
# TRAIN + VALIDATE BPR CONFIGURATIONS
# ============================================================

print("=" * 70)
print("BPR MODEL SELECTION USING VALIDATION")
print("=" * 70)


validation_results = []

trained_models = {}


for config in BPR_CONFIGS:

    print(
        "\n" + "=" * 70
    )

    print(
        "TRAIN:",
        config["name"]
    )

    print(
        "=" * 70
    )


    start_time = time.time()


    bpr_model = (
        BayesianPersonalizedRanking(

            factors=
                config[
                    "factors"
                ],

            regularization=
                config[
                    "regularization"
                ],

            learning_rate=
                config[
                    "learning_rate"
                ],

            iterations=
                config[
                    "iterations"
                ],

            random_state=
                RANDOM_STATE,

            verify_negative_samples=
                True
        )
    )


    bpr_model.fit(
        user_item_matrix,
        show_progress=True
    )


    elapsed = (
        time.time()
        - start_time
    )


    print(
        f"\nTraining time: "
        f"{elapsed:.2f} sec"
    )


    result = evaluate_bpr(
        bpr_model,
        validation,
        ks=(5, 10, 20)
    )


    result[
        "model"
    ] = config["name"]

    result[
        "factors"
    ] = config["factors"]

    result[
        "regularization"
    ] = config[
        "regularization"
    ]

    result[
        "iterations"
    ] = config[
        "iterations"
    ]


    validation_results.append(
        result
    )


    trained_models[
        config["name"]
    ] = bpr_model


validation_results = pd.concat(
    validation_results,
    ignore_index=True
)


display(
    validation_results[
        [
            "model",
            "K",
            "users",
            "HitRate",
            "Recall",
            "NDCG",
            "MRR"
        ]
    ].round(6)
)

BPR MODEL SELECTION USING VALIDATION

TRAIN: BPR_F32_R01


  0%|          | 0/50 [00:00<?, ?it/s]


Training time: 1.11 sec
Evaluable users: 26,745 / 26,903 (99.41%)

TRAIN: BPR_F64_R01


  0%|          | 0/50 [00:00<?, ?it/s]


Training time: 1.46 sec
Evaluable users: 26,745 / 26,903 (99.41%)

TRAIN: BPR_F64_R05


  0%|          | 0/50 [00:00<?, ?it/s]


Training time: 1.42 sec
Evaluable users: 26,745 / 26,903 (99.41%)


,model,K,users,HitRate,Recall,NDCG,MRR
0,BPR_F32_R01,5,26745,0.002617,0.002617,0.001666,0.001352
1,BPR_F32_R01,10,26745,0.004674,0.004674,0.002312,0.001608
2,BPR_F32_R01,20,26745,0.007366,0.007366,0.002985,0.001789
3,BPR_F64_R01,5,26745,0.003216,0.003216,0.001982,0.001579
4,BPR_F64_R01,10,26745,0.005048,0.005048,0.002559,0.001809
5,BPR_F64_R01,20,26745,0.008712,0.008712,0.003482,0.002060
6,BPR_F64_R05,5,26745,0.004412,0.004412,0.002655,0.002082
7,BPR_F64_R05,10,26745,0.006768,0.006768,0.003413,0.002393
8,BPR_F64_R05,20,26745,0.010656,0.010656,0.004389,0.002657


# Chọn BPR bằng Validation NDCG@10

In [48]:
# ============================================================
# STEP 6 - CELL 19
# SELECT BEST MODEL ON VALIDATION
# ============================================================

selection_table = (
    validation_results[
        validation_results[
            "K"
        ] == 10
    ]
    .sort_values(
        [
            "NDCG",
            "Recall"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 70)
print("MODEL SELECTION - VALIDATION @10")
print("=" * 70)

display(
    selection_table[
        [
            "model",
            "HitRate",
            "Recall",
            "NDCG",
            "MRR"
        ]
    ].round(6)
)


BEST_MODEL_NAME = (
    selection_table
    .iloc[0][
        "model"
    ]
)


best_model = (
    trained_models[
        BEST_MODEL_NAME
    ]
)


print(
    "\nSelected model:",
    BEST_MODEL_NAME
)

print(
    "Selection criterion:"
    " highest Validation NDCG@10"
)

MODEL SELECTION - VALIDATION @10


,model,HitRate,Recall,NDCG,MRR
0,BPR_F64_R05,0.006768,0.006768,0.003413,0.002393
1,BPR_F64_R01,0.005048,0.005048,0.002559,0.001809
2,BPR_F32_R01,0.004674,0.004674,0.002312,0.001608



Selected model: BPR_F64_R05
Selection criterion: highest Validation NDCG@10


# VALIDATION BASELINE COMPARISON


In [49]:
# ============================================================
# STEP 6 - CELL 20
# VALIDATION BASELINE COMPARISON - ROBUST VERSION
# ============================================================

print("=" * 70)
print("VALIDATION COMPARISON")
print("=" * 70)


# ------------------------------------------------------------
# 1. Recompute Popularity validation results
# ------------------------------------------------------------

print("\n[1] Evaluating Popularity baseline...")

pop_valid_results = evaluate_popularity(
    validation,
    ks=(5, 10, 20)
)


# ------------------------------------------------------------
# 2. Check BPR validation results
# ------------------------------------------------------------

if "validation_results" not in globals():

    raise RuntimeError(
        "validation_results does not exist. "
        "Please run Cell 18 first."
    )


if "BEST_MODEL_NAME" not in globals():

    # Automatically select best model using Validation NDCG@10

    selection_table = (
        validation_results[
            validation_results["K"] == 10
        ]
        .sort_values(
            ["NDCG", "Recall"],
            ascending=False
        )
        .reset_index(drop=True)
    )

    BEST_MODEL_NAME = (
        selection_table.iloc[0]["model"]
    )


print(
    "\n[2] Selected BPR model:",
    BEST_MODEL_NAME
)


# ------------------------------------------------------------
# 3. Extract best BPR validation results
# ------------------------------------------------------------

best_bpr_valid = (
    validation_results[
        validation_results["model"]
        == BEST_MODEL_NAME
    ]
    .copy()
)


# ------------------------------------------------------------
# 4. Standardize columns
# ------------------------------------------------------------

comparison_columns = [
    "model",
    "K",
    "users",
    "HitRate",
    "Recall",
    "NDCG",
    "MRR"
]


pop_compare = (
    pop_valid_results[
        comparison_columns
    ]
    .copy()
)


bpr_compare = (
    best_bpr_valid[
        comparison_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# 5. Combine
# ------------------------------------------------------------

comparison_valid = pd.concat(
    [
        pop_compare,
        bpr_compare
    ],
    ignore_index=True
)


comparison_valid = (
    comparison_valid
    .sort_values(
        ["K", "model"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("POPULARITY VS BPR - VALIDATION")
print("=" * 70)

display(
    comparison_valid.round(6)
)


# ------------------------------------------------------------
# 7. Focus on K = 10
# ------------------------------------------------------------

comparison_k10 = (
    comparison_valid[
        comparison_valid["K"] == 10
    ]
    .copy()
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("VALIDATION @10")
print("=" * 70)

display(
    comparison_k10.round(6)
)


print("\n✅ Validation comparison completed")

VALIDATION COMPARISON

[1] Evaluating Popularity baseline...

[2] Selected BPR model: BPR_F64_R05

POPULARITY VS BPR - VALIDATION


,model,K,users,HitRate,Recall,NDCG,MRR
0,BPR_F64_R05,5,26745,0.004412,0.004412,0.002655,0.002082
1,Popularity,5,26745,0.015143,0.015143,0.009694,0.007925
2,BPR_F64_R05,10,26745,0.006768,0.006768,0.003413,0.002393
3,Popularity,10,26745,0.026173,0.026173,0.013180,0.009314
4,BPR_F64_R05,20,26745,0.010656,0.010656,0.004389,0.002657
5,Popularity,20,26745,0.037839,0.037839,0.016110,0.010109



VALIDATION @10


,model,K,users,HitRate,Recall,NDCG,MRR
0,BPR_F64_R05,10,26745,0.006768,0.006768,0.003413,0.002393
1,Popularity,10,26745,0.026173,0.026173,0.013180,0.009314



✅ Validation comparison completed


In [55]:
# ============================================================
# STEP 6 - CELL 20B
# POSITIVE BPR MATRIX - RATING >= 4
# ============================================================

print("=" * 70)
print("POSITIVE BPR - BUILD MATRIX")
print("=" * 70)

POSITIVE_THRESHOLD = 4.0

positive_train = (
    train[
        pd.to_numeric(
            train["rating"],
            errors="coerce"
        ) >= POSITIVE_THRESHOLD
    ]
    .copy()
)

print("Original train interactions :", f"{len(train):,}")
print("Positive interactions       :", f"{len(positive_train):,}")
print(
    "Retained                    :",
    f"{len(positive_train)/len(train)*100:.2f}%"
)

print(
    "\nUsers with positives:",
    f"{positive_train['user_id'].nunique():,}"
)

print(
    "Products with positives:",
    f"{positive_train['product_id'].nunique():,}"
)

# ------------------------------------------------------------
# Use SAME mappings as original CF model
# ------------------------------------------------------------

positive_train["user_idx"] = (
    positive_train["user_id"]
    .map(user_to_idx)
)

positive_train["item_idx"] = (
    positive_train["product_id"]
    .map(item_to_idx)
)

positive_train = (
    positive_train[
        positive_train["user_idx"].notna()
        &
        positive_train["item_idx"].notna()
    ]
    .copy()
)

positive_user_item_matrix = csr_matrix(
    (
        np.ones(
            len(positive_train),
            dtype=np.float32
        ),
        (
            positive_train["user_idx"].astype(int),
            positive_train["item_idx"].astype(int)
        )
    ),
    shape=user_item_matrix.shape,
    dtype=np.float32
)

print("\nMatrix shape :", positive_user_item_matrix.shape)
print("Non-zero     :", f"{positive_user_item_matrix.nnz:,}")

positive_users = set(
    positive_train["user_id"].unique()
)

print(
    "Users without positive train history:",
    f"{len(user_to_idx) - len(positive_users):,}"
)

print("\n✅ Positive matrix ready")

POSITIVE BPR - BUILD MATRIX
Original train interactions : 205,621
Positive interactions       : 172,854
Retained                    : 84.06%

Users with positives: 26,641
Products with positives: 15,449

Matrix shape : (26903, 15546)
Non-zero     : 172,854
Users without positive train history: 262

✅ Positive matrix ready


In [56]:
# ============================================================
# STEP 6 - CELL 20C
# POSITIVE BPR RECOMMENDER
# ============================================================

def recommend_positive_bpr(
    model,
    user_id,
    k=10
):
    if user_id not in user_to_idx:
        return []

    # No positive preference history
    if user_id not in positive_users:
        return []

    user_idx = user_to_idx[user_id]

    ids, scores = model.recommend(
        userid=user_idx,
        user_items=positive_user_item_matrix[user_idx],
        N=k,
        filter_already_liked_items=True
    )

    return idx_to_item[ids].tolist()


def evaluate_positive_bpr(
    model,
    holdout_df,
    ks=(5, 10, 20)
):
    df = holdout_df.copy()

    # Target must exist in training item catalog
    df["known_item"] = df["product_id"].isin(item_to_idx)

    # User must have positive training history
    df["positive_user"] = df["user_id"].isin(positive_users)

    evaluable = df[
        df["known_item"]
        &
        df["positive_user"]
    ].copy()

    print(
        f"Evaluable users: {len(evaluable):,} / {len(df):,} "
        f"({len(evaluable)/len(df)*100:.2f}%)"
    )

    max_k = max(ks)

    store = {
        k: {
            "hit": [],
            "recall": [],
            "ndcg": [],
            "mrr": []
        }
        for k in ks
    }

    for row in evaluable.itertuples():

        ranked = recommend_positive_bpr(
            model,
            row.user_id,
            k=max_k
        )

        for k in ks:

            metrics = evaluate_ranked_items(
                ranked,
                row.product_id,
                k
            )

            for metric, value in metrics.items():
                store[k][metric].append(value)

    rows = []

    for k in ks:

        rows.append({
            "K": k,
            "users": len(evaluable),
            "HitRate": np.mean(store[k]["hit"]),
            "Recall": np.mean(store[k]["recall"]),
            "NDCG": np.mean(store[k]["ndcg"]),
            "MRR": np.mean(store[k]["mrr"])
        })

    return pd.DataFrame(rows)


print("✅ Positive-BPR functions ready")

✅ Positive-BPR functions ready


In [57]:
# ============================================================
# STEP 6 - CELL 20D
# TRAIN POSITIVE-BPR CONFIGURATIONS
# ============================================================

POSITIVE_BPR_CONFIGS = [
    {
        "name": "PosBPR_F32_R01",
        "factors": 32,
        "regularization": 0.01,
        "iterations": 50,
        "learning_rate": 0.05
    },
    {
        "name": "PosBPR_F64_R01",
        "factors": 64,
        "regularization": 0.01,
        "iterations": 50,
        "learning_rate": 0.05
    },
    {
        "name": "PosBPR_F64_R05",
        "factors": 64,
        "regularization": 0.05,
        "iterations": 50,
        "learning_rate": 0.05
    }
]

positive_validation_results = []
positive_models = {}

for config in POSITIVE_BPR_CONFIGS:

    print("\n" + "=" * 70)
    print("TRAIN:", config["name"])
    print("=" * 70)

    model_pos = BayesianPersonalizedRanking(
        factors=config["factors"],
        regularization=config["regularization"],
        learning_rate=config["learning_rate"],
        iterations=config["iterations"],
        random_state=RANDOM_STATE,
        verify_negative_samples=True
    )

    start = time.time()

    model_pos.fit(
        positive_user_item_matrix,
        show_progress=True
    )

    print(
        f"\nTraining time: {time.time()-start:.2f} sec"
    )

    result = evaluate_positive_bpr(
        model_pos,
        validation,
        ks=(5, 10, 20)
    )

    result["model"] = config["name"]
    result["factors"] = config["factors"]
    result["regularization"] = config["regularization"]

    positive_validation_results.append(result)

    positive_models[
        config["name"]
    ] = model_pos


positive_validation_results = pd.concat(
    positive_validation_results,
    ignore_index=True
)

print("\n" + "=" * 70)
print("POSITIVE-BPR VALIDATION RESULTS")
print("=" * 70)

display(
    positive_validation_results[
        [
            "model",
            "K",
            "users",
            "HitRate",
            "Recall",
            "NDCG",
            "MRR"
        ]
    ].round(6)
)


TRAIN: PosBPR_F32_R01


  0%|          | 0/50 [00:00<?, ?it/s]


Training time: 0.90 sec
Evaluable users: 26,486 / 26,903 (98.45%)

TRAIN: PosBPR_F64_R01


  0%|          | 0/50 [00:00<?, ?it/s]


Training time: 1.25 sec
Evaluable users: 26,486 / 26,903 (98.45%)

TRAIN: PosBPR_F64_R05


  0%|          | 0/50 [00:00<?, ?it/s]


Training time: 1.21 sec
Evaluable users: 26,486 / 26,903 (98.45%)

POSITIVE-BPR VALIDATION RESULTS


,model,K,users,HitRate,Recall,NDCG,MRR
0,PosBPR_F32_R01,5,26486,0.002756,0.002756,0.001746,0.001418
1,PosBPR_F32_R01,10,26486,0.004229,0.004229,0.002223,0.001615
2,PosBPR_F32_R01,20,26486,0.007702,0.007702,0.003092,0.001849
3,PosBPR_F64_R01,5,26486,0.003058,0.003058,0.002045,0.001710
4,PosBPR_F64_R01,10,26486,0.004795,0.004795,0.002597,0.001932
5,PosBPR_F64_R01,20,26486,0.008117,0.008117,0.003422,0.002151
6,PosBPR_F64_R05,5,26486,0.003625,0.003625,0.002451,0.002063
7,PosBPR_F64_R05,10,26486,0.006230,0.006230,0.003287,0.002404
8,PosBPR_F64_R05,20,26486,0.010345,0.010345,0.004315,0.002679


In [58]:
# ============================================================
# STEP 6 - CELL 20E
# SELECT BEST POSITIVE-BPR
# ============================================================

positive_selection = (
    positive_validation_results[
        positive_validation_results["K"] == 10
    ]
    .sort_values(
        ["NDCG", "Recall"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("POSITIVE-BPR MODEL SELECTION @10")
print("=" * 70)

display(
    positive_selection[
        [
            "model",
            "users",
            "HitRate",
            "Recall",
            "NDCG",
            "MRR"
        ]
    ].round(6)
)

BEST_POSITIVE_MODEL_NAME = (
    positive_selection.iloc[0]["model"]
)

best_positive_model = (
    positive_models[
        BEST_POSITIVE_MODEL_NAME
    ]
)

print(
    "\nSelected Positive-BPR:",
    BEST_POSITIVE_MODEL_NAME
)

POSITIVE-BPR MODEL SELECTION @10


,model,users,HitRate,Recall,NDCG,MRR
0,PosBPR_F64_R05,26486,0.006230,0.006230,0.003287,0.002404
1,PosBPR_F64_R01,26486,0.004795,0.004795,0.002597,0.001932
2,PosBPR_F32_R01,26486,0.004229,0.004229,0.002223,0.001615



Selected Positive-BPR: PosBPR_F64_R05


In [59]:
# ============================================================
# STEP 6 - CELL 20F
# FINAL VALIDATION COMPARISON
# ============================================================

original_best = (
    validation_results[
        (validation_results["model"] == BEST_MODEL_NAME)
        &
        (validation_results["K"] == 10)
    ][
        [
            "model",
            "K",
            "users",
            "HitRate",
            "Recall",
            "NDCG",
            "MRR"
        ]
    ]
    .copy()
)

positive_best = (
    positive_validation_results[
        (
            positive_validation_results["model"]
            == BEST_POSITIVE_MODEL_NAME
        )
        &
        (
            positive_validation_results["K"]
            == 10
        )
    ][
        [
            "model",
            "K",
            "users",
            "HitRate",
            "Recall",
            "NDCG",
            "MRR"
        ]
    ]
    .copy()
)

pop_k10 = (
    pop_valid_results[
        pop_valid_results["K"] == 10
    ][
        [
            "model",
            "K",
            "users",
            "HitRate",
            "Recall",
            "NDCG",
            "MRR"
        ]
    ]
    .copy()
)

final_validation_comparison = pd.concat(
    [
        pop_k10,
        original_best,
        positive_best
    ],
    ignore_index=True
)

print("=" * 70)
print("FINAL MODEL COMPARISON - VALIDATION @10")
print("=" * 70)

display(
    final_validation_comparison.round(6)
)

print("\nImportant:")
print(
    "Compare the 'users' column before interpreting metrics."
)
print(
    "Test set remains untouched."
)

FINAL MODEL COMPARISON - VALIDATION @10


,model,K,users,HitRate,Recall,NDCG,MRR
0,Popularity,10,26745,0.026173,0.026173,0.013180,0.009314
1,BPR_F64_R05,10,26745,0.006768,0.006768,0.003413,0.002393
2,PosBPR_F64_R05,10,26486,0.006230,0.006230,0.003287,0.002404



Important:
Compare the 'users' column before interpreting metrics.
Test set remains untouched.


In [60]:
# ============================================================
# STEP 6 - CELL 21
# FINAL TEST EVALUATION
# TEST SET IS OPENED ONLY AFTER VALIDATION MODEL SELECTION
# ============================================================

print("=" * 70)
print("FINAL TEST EVALUATION")
print("=" * 70)

print("Selected CF model :", BEST_MODEL_NAME)
print("Selection basis   : Validation NDCG@10")
print("Test interactions :", f"{len(test):,}")


# ------------------------------------------------------------
# 1. Rebuild test known-item flag
# ------------------------------------------------------------

test = test.copy()

test["known_item"] = (
    test["product_id"]
    .isin(item_to_idx)
)

known_test = test[
    test["known_item"]
].copy()

print("\nKnown-item test users:",
      f"{len(known_test):,}",
      f"({len(known_test)/len(test)*100:.2f}%)")

print("Cold-item test users :",
      f"{(~test['known_item']).sum():,}",
      f"({(~test['known_item']).mean()*100:.2f}%)")


# ------------------------------------------------------------
# 2. Popularity
# ------------------------------------------------------------

print("\n[1] Popularity baseline")

pop_test_results = evaluate_popularity(
    test,
    ks=(5, 10, 20)
)


# ------------------------------------------------------------
# 3. Selected BPR
# ------------------------------------------------------------

print("\n[2] Selected BPR")

bpr_test_results = evaluate_bpr(
    best_model,
    test,
    ks=(5, 10, 20)
)

bpr_test_results["model"] = (
    BEST_MODEL_NAME
)


# ------------------------------------------------------------
# 4. Combine
# ------------------------------------------------------------

test_results = pd.concat(
    [
        pop_test_results[
            [
                "model",
                "K",
                "users",
                "HitRate",
                "Recall",
                "NDCG",
                "MRR"
            ]
        ],

        bpr_test_results[
            [
                "model",
                "K",
                "users",
                "HitRate",
                "Recall",
                "NDCG",
                "MRR"
            ]
        ]
    ],
    ignore_index=True
)


test_results = (
    test_results
    .sort_values(
        ["K", "model"]
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("FINAL TEST RESULTS - KNOWN ITEMS")
print("=" * 70)

display(
    test_results.round(6)
)


# ------------------------------------------------------------
# 5. Focus @10
# ------------------------------------------------------------

test_k10 = (
    test_results[
        test_results["K"] == 10
    ]
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("FINAL TEST @10")
print("=" * 70)

display(
    test_k10.round(6)
)

print("\n✅ Final test evaluation completed")

FINAL TEST EVALUATION
Selected CF model : BPR_F64_R05
Selection basis   : Validation NDCG@10
Test interactions : 26,903

Known-item test users: 26,585 (98.82%)
Cold-item test users : 318 (1.18%)

[1] Popularity baseline

[2] Selected BPR
Evaluable users: 26,585 / 26,903 (98.82%)

FINAL TEST RESULTS - KNOWN ITEMS


,model,K,users,HitRate,Recall,NDCG,MRR
0,BPR_F64_R05,5,26585,0.003009,0.003009,0.001864,0.001492
1,Popularity,5,26585,0.010118,0.010118,0.006581,0.005432
2,BPR_F64_R05,10,26585,0.004589,0.004589,0.002375,0.001703
3,Popularity,10,26585,0.018695,0.018695,0.009259,0.006481
4,BPR_F64_R05,20,26585,0.006996,0.006996,0.002980,0.001867
5,Popularity,20,26585,0.027158,0.027158,0.011391,0.007062



FINAL TEST @10


,model,K,users,HitRate,Recall,NDCG,MRR
0,BPR_F64_R05,10,26585,0.004589,0.004589,0.002375,0.001703
1,Popularity,10,26585,0.018695,0.018695,0.009259,0.006481



✅ Final test evaluation completed


In [61]:
# ============================================================
# STEP 6 - CELL 22
# FULL TEST POPULATION
# ============================================================

def evaluate_model_full_test(
    model,
    holdout_df,
    ks=(5, 10, 20)
):

    max_k = max(ks)

    store = {
        k: {
            "hit": [],
            "recall": [],
            "ndcg": [],
            "mrr": []
        }
        for k in ks
    }

    cold_targets = 0

    for row in holdout_df.itertuples():

        # CF cannot represent target
        if row.product_id not in item_to_idx:

            ranked = []
            cold_targets += 1

        else:

            ranked = recommend_bpr(
                model,
                row.user_id,
                k=max_k
            )

        for k in ks:

            metrics = evaluate_ranked_items(
                ranked,
                row.product_id,
                k
            )

            for metric, value in metrics.items():

                store[k][metric].append(
                    value
                )

    rows = []

    for k in ks:

        rows.append({
            "model": BEST_MODEL_NAME,
            "population": "all_test_users",
            "K": k,
            "users": len(holdout_df),

            "HitRate": np.mean(
                store[k]["hit"]
            ),

            "Recall": np.mean(
                store[k]["recall"]
            ),

            "NDCG": np.mean(
                store[k]["ndcg"]
            ),

            "MRR": np.mean(
                store[k]["mrr"]
            )
        })

    return (
        pd.DataFrame(rows),
        cold_targets
    )


full_test_results, cold_test_targets = (
    evaluate_model_full_test(
        best_model,
        test,
        ks=(5, 10, 20)
    )
)


print("=" * 70)
print("BPR - FULL TEST POPULATION")
print("=" * 70)

print(
    "All test users :",
    f"{len(test):,}"
)

print(
    "Cold targets   :",
    f"{cold_test_targets:,}"
)

print(
    "Cold rate      :",
    f"{cold_test_targets/len(test)*100:.2f}%"
)

display(
    full_test_results.round(6)
)

BPR - FULL TEST POPULATION
All test users : 26,903
Cold targets   : 318
Cold rate      : 1.18%


,model,population,K,users,HitRate,Recall,NDCG,MRR
0,BPR_F64_R05,all_test_users,5,26903,0.002974,0.002974,0.001842,0.001474
1,BPR_F64_R05,all_test_users,10,26903,0.004535,0.004535,0.002347,0.001683
2,BPR_F64_R05,all_test_users,20,26903,0.006914,0.006914,0.002945,0.001845


# FINAL TEST EVALUATION


In [50]:
# ============================================================
# STEP 6 - CELL 21
# FINAL TEST EVALUATION
# ============================================================

print("=" * 70)
print("FINAL TEST EVALUATION")
print("=" * 70)


pop_test_results = (
    evaluate_popularity(
        test,
        ks=(5, 10, 20)
    )
)


bpr_test_results = (
    evaluate_bpr(
        best_model,
        test,
        ks=(5, 10, 20)
    )
)


bpr_test_results[
    "model"
] = BEST_MODEL_NAME


test_results = pd.concat(
    [
        pop_test_results,
        bpr_test_results
    ],
    ignore_index=True
)


display(
    test_results[
        [
            "model",
            "K",
            "users",
            "HitRate",
            "Recall",
            "NDCG",
            "MRR"
        ]
    ].round(6)
)

FINAL TEST EVALUATION
Evaluable users: 26,585 / 26,903 (98.82%)


,model,K,users,HitRate,Recall,NDCG,MRR
0,Popularity,5,26585,0.010118,0.010118,0.006581,0.005432
1,Popularity,10,26585,0.018695,0.018695,0.009259,0.006481
2,Popularity,20,26585,0.027158,0.027158,0.011391,0.007062
3,BPR_F64_R05,5,26585,0.003009,0.003009,0.001864,0.001492
4,BPR_F64_R05,10,26585,0.004589,0.004589,0.002375,0.001703
5,BPR_F64_R05,20,26585,0.006996,0.006996,0.002980,0.001867


# FULL TEST POPULATION EVALUATION


In [51]:
# ============================================================
# STEP 6 - CELL 22
# FULL TEST POPULATION EVALUATION
# ============================================================

def evaluate_bpr_full_population(
    model,
    holdout_df,
    ks=(5, 10, 20)
):

    max_k = max(
        ks
    )

    metric_store = {

        k: {
            "hit": [],
            "recall": [],
            "ndcg": [],
            "mrr": []
        }

        for k in ks
    }


    for row in holdout_df.itertuples():

        if (
            row.product_id
            not in item_to_idx
        ):

            ranked = []

        else:

            ranked = recommend_bpr(
                model,
                row.user_id,
                k=max_k
            )


        for k in ks:

            metrics = (
                evaluate_ranked_items(
                    ranked,
                    row.product_id,
                    k
                )
            )

            for metric, value in metrics.items():

                metric_store[
                    k
                ][
                    metric
                ].append(
                    value
                )


    rows = []

    for k in ks:

        rows.append({

            "model":
                BEST_MODEL_NAME,

            "population":
                "all_test_users",

            "K":
                k,

            "users":
                len(holdout_df),

            "HitRate":
                np.mean(
                    metric_store[k][
                        "hit"
                    ]
                ),

            "Recall":
                np.mean(
                    metric_store[k][
                        "recall"
                    ]
                ),

            "NDCG":
                np.mean(
                    metric_store[k][
                        "ndcg"
                    ]
                ),

            "MRR":
                np.mean(
                    metric_store[k][
                        "mrr"
                    ]
                )
        })


    return pd.DataFrame(
        rows
    )


full_test_results = (
    evaluate_bpr_full_population(
        best_model,
        test
    )
)


print("=" * 70)
print("BPR - ALL TEST USERS")
print("=" * 70)

display(
    full_test_results.round(6)
)

BPR - ALL TEST USERS


,model,population,K,users,HitRate,Recall,NDCG,MRR
0,BPR_F64_R05,all_test_users,5,26903,0.002974,0.002974,0.001842,0.001474
1,BPR_F64_R05,all_test_users,10,26903,0.004535,0.004535,0.002347,0.001683
2,BPR_F64_R05,all_test_users,20,26903,0.006914,0.006914,0.002945,0.001845


# Recommendation Example

In [52]:
# ============================================================
# STEP 6 - CELL 23
# SAMPLE PERSONALIZED RECOMMENDATIONS
# ============================================================

sample_user = (
    test[
        test[
            "known_item"
        ]
    ]
    .iloc[0][
        "user_id"
    ]
)


history = (
    train[
        train[
            "user_id"
        ] == sample_user
    ]
    .sort_values(
        "timestamp"
    )
)


recommendations = (
    recommend_bpr(
        best_model,
        sample_user,
        k=10
    )
)


target = (
    test[
        test[
            "user_id"
        ] == sample_user
    ]
    .iloc[0][
        "product_id"
    ]
)


print("=" * 70)

print(
    "USER:",
    sample_user
)

print("=" * 70)


print("\nTRAIN HISTORY")

display(
    history[
        [
            "product_id",
            "rating",
            "datetime"
        ]
    ]
)


print(
    "\nTEST TARGET:",
    target
)


print(
    "\nTOP-10 RECOMMENDATIONS:"
)

for rank, product_id in enumerate(
    recommendations,
    start=1
):

    marker = (
        " <-- TEST TARGET"
        if product_id == target
        else ""
    )

    print(
        f"{rank:02d}. "
        f"{product_id}"
        f"{marker}"
    )

USER: amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA

TRAIN HISTORY


,product_id,rating,datetime
0,amazon_B0007LCLPE,2.0,2015-07-11 12:25:02
1,amazon_B08KHM6QJB,4.0,2016-05-09 20:09:52
2,amazon_B00U3FPN4U,5.0,2016-05-09 20:22:15
3,amazon_B00BIDK1WU,3.0,2016-05-09 20:31:44
4,amazon_B0787HCKR2,4.0,2016-05-09 20:34:15



TEST TARGET: amazon_B0C5MBN688

TOP-10 RECOMMENDATIONS:
01. amazon_B002TZUOC8
02. amazon_B0862W5BQ8
03. amazon_B08ZDF3WT3
04. amazon_B07BNQ321S
05. amazon_B015YF5YIS
06. amazon_B0BYGS3RZ7
07. amazon_B005JBWX46
08. amazon_B07C5QRJZD
09. amazon_B00D7CWSCG
10. amazon_B00HPMCO6O


# Generate CF Product Scores / Factors

In [53]:
# ============================================================
# STEP 6 - CELL 24
# PREPARE CF ARTIFACTS
# ============================================================

print("=" * 70)
print("PREPARE CF ARTIFACTS")
print("=" * 70)


cf_product_metadata = pd.DataFrame({

    "item_idx":
        np.arange(
            len(idx_to_item)
        ),

    "product_id":
        idx_to_item
})


pop_lookup = (
    item_popularity
    .set_index(
        "product_id"
    )
)


cf_product_metadata[
    "interaction_count"
] = (
    cf_product_metadata[
        "product_id"
    ]
    .map(
        pop_lookup[
            "interaction_count"
        ]
    )
    .fillna(0)
)


cf_product_metadata[
    "avg_rating"
] = (
    cf_product_metadata[
        "product_id"
    ]
    .map(
        pop_lookup[
            "avg_rating"
        ]
    )
)


print(
    "CF products:",
    f"{len(cf_product_metadata):,}"
)

display(
    cf_product_metadata.head()
)

PREPARE CF ARTIFACTS
CF products: 15,546


,item_idx,product_id,interaction_count,avg_rating
0,0,amazon_B0007LCLPE,5,4.400000
1,1,amazon_B08KHM6QJB,20,4.350000
2,2,amazon_B00U3FPN4U,350,4.200000
3,3,amazon_B00BIDK1WU,8,4.750000
4,4,amazon_B0787HCKR2,45,4.711111


# FINAL CF AUDIT


In [54]:
# ============================================================
# STEP 6 - CELL 25
# FINAL CF AUDIT
# ============================================================

print("=" * 70)
print("FINAL CF AUDIT")
print("=" * 70)


checks = {

    "train_users_valid":
        train[
            "user_id"
        ].value_counts().min()
        >= 3,

    "validation_one_per_user":
        validation[
            "user_id"
        ].value_counts().eq(1).all(),

    "test_one_per_user":
        test[
            "user_id"
        ].value_counts().eq(1).all(),

    "matrix_users":
        user_item_matrix.shape[0]
        == len(user_to_idx),

    "matrix_items":
        user_item_matrix.shape[1]
        == len(item_to_idx),

    "model_user_factors":
        best_model.user_factors.shape[0]
        == len(user_to_idx),

    "model_item_factors":
        best_model.item_factors.shape[0]
        == len(item_to_idx),

    "temporal_train_valid":
        train_before_valid,

    "temporal_valid_test":
        valid_before_test
}


for name, status in checks.items():

    print(
        ("✅" if status else "❌"),
        f"{name:<30}:",
        status
    )


print("\n" + "-" * 70)

print(
    "Train interactions:",
    f"{len(train):,}"
)

print(
    "Validation:",
    f"{len(validation):,}"
)

print(
    "Test:",
    f"{len(test):,}"
)

print(
    "Train users:",
    f"{len(user_to_idx):,}"
)

print(
    "Train products:",
    f"{len(item_to_idx):,}"
)

print(
    "Selected model:",
    BEST_MODEL_NAME
)


assert all(
    checks.values()
)

print(
    "\n✅ Collaborative Filtering checks passed"
)

FINAL CF AUDIT
✅ train_users_valid             : True
✅ validation_one_per_user       : True
✅ test_one_per_user             : True
✅ matrix_users                  : True
✅ matrix_items                  : True
✅ model_user_factors            : True
✅ model_item_factors            : True
✅ temporal_train_valid          : True
✅ temporal_valid_test           : True

----------------------------------------------------------------------
Train interactions: 205,621
Validation: 26,903
Test: 26,903
Train users: 26,903
Train products: 15,546
Selected model: BPR_F64_R05

✅ Collaborative Filtering checks passed


# SAVE STEP 6 ARTIFACTS


In [62]:
# ============================================================
# STEP 6 - CELL 24
# SAVE FINAL CF ARTIFACTS
# ============================================================

import os
import json
import pickle

OUTPUT_DIR = "/kaggle/working/step6"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("SAVE FINAL CF ARTIFACTS")
print("=" * 70)


# ------------------------------------------------------------
# 1. Save selected BPR model
# ------------------------------------------------------------

model_path = os.path.join(
    OUTPUT_DIR,
    "bpr_model.pkl"
)

with open(model_path, "wb") as f:
    pickle.dump(best_model, f)


# ------------------------------------------------------------
# 2. Save mappings
# ------------------------------------------------------------

mapping_path = os.path.join(
    OUTPUT_DIR,
    "cf_mappings.pkl"
)

with open(mapping_path, "wb") as f:
    pickle.dump(
        {
            "user_to_idx": user_to_idx,
            "item_to_idx": item_to_idx,
            "idx_to_user": idx_to_user,
            "idx_to_item": idx_to_item
        },
        f
    )


# ------------------------------------------------------------
# 3. Save train interactions
# Needed later for user history / Hybrid
# ------------------------------------------------------------

train_save = train[
    [
        "user_id",
        "product_id",
        "rating",
        "timestamp"
    ]
].copy()

train_save.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cf_train.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 4. Save validation/test results
# ------------------------------------------------------------

validation_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "bpr_validation_results.csv"
    ),
    index=False
)

test_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cf_test_results.csv"
    ),
    index=False
)

full_test_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "bpr_full_test_results.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 5. Save configuration
# ------------------------------------------------------------

selected_config = next(
    x
    for x in BPR_CONFIGS
    if x["name"] == BEST_MODEL_NAME
)

config = {
    "model_type": "BayesianPersonalizedRanking",

    "selected_model": BEST_MODEL_NAME,

    "selection_metric": "Validation NDCG@10",

    "factors": int(
        selected_config["factors"]
    ),

    "regularization": float(
        selected_config["regularization"]
    ),

    "iterations": int(
        selected_config["iterations"]
    ),

    "learning_rate": float(
        selected_config["learning_rate"]
    ),

    "train_users": int(
        len(user_to_idx)
    ),

    "train_products": int(
        len(item_to_idx)
    ),

    "train_interactions": int(
        len(train)
    ),

    "test_users": int(
        len(test)
    ),

    "test_known_targets": int(
        test["product_id"]
        .isin(item_to_idx)
        .sum()
    ),

    "test_cold_targets": int(
        (~test["product_id"]
         .isin(item_to_idx))
        .sum()
    )
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "cf_config.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\nSaved:")
for file in sorted(os.listdir(OUTPUT_DIR)):

    path = os.path.join(
        OUTPUT_DIR,
        file
    )

    size_mb = (
        os.path.getsize(path)
        / 1024**2
    )

    print(
        f"{file:<32} "
        f"{size_mb:>8.2f} MB"
    )

print(
    "\nOutput directory:",
    OUTPUT_DIR
)

print("\n✅ Step 6 artifacts saved")

SAVE FINAL CF ARTIFACTS

Saved:
bpr_full_test_results.csv            0.00 MB
bpr_model.pkl                       10.53 MB
bpr_validation_results.csv           0.00 MB
cf_config.json                       0.00 MB
cf_mappings.pkl                      1.59 MB
cf_test_results.csv                  0.00 MB
cf_train.csv                        14.12 MB

Output directory: /kaggle/working/step6

✅ Step 6 artifacts saved


In [63]:
# ============================================================
# STEP 6 - CELL 25
# FINAL CF AUDIT
# ============================================================

print("=" * 70)
print("FINAL STEP 6 AUDIT")
print("=" * 70)

checks = {
    "selected_model_exists":
        best_model is not None,

    "selected_model_name":
        BEST_MODEL_NAME == "BPR_F64_R05",

    "train_users_consistent":
        len(user_to_idx) == train["user_id"].nunique(),

    "train_products_consistent":
        len(item_to_idx) == train["product_id"].nunique(),

    "validation_results_exist":
        len(validation_results) > 0,

    "test_results_exist":
        len(test_results) > 0,

    "full_test_results_exist":
        len(full_test_results) > 0,

    "test_timestamp_complete":
        test["timestamp"].notna().all()
}

for key, value in checks.items():

    icon = "✅" if value else "❌"

    print(
        f"{icon} {key:<32}: {value}"
    )


assert all(checks.values())


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

bpr_test10 = test_results[
    (test_results["model"] == BEST_MODEL_NAME)
    &
    (test_results["K"] == 10)
].iloc[0]

pop_test10 = test_results[
    (test_results["model"] == "Popularity")
    &
    (test_results["K"] == 10)
].iloc[0]

full_bpr10 = full_test_results[
    full_test_results["K"] == 10
].iloc[0]


print("\n" + "-" * 70)
print("STEP 6 FINAL SUMMARY")
print("-" * 70)

print(
    f"Selected CF model : {BEST_MODEL_NAME}"
)

print(
    f"CF train users    : {len(user_to_idx):,}"
)

print(
    f"CF train products : {len(item_to_idx):,}"
)

print(
    f"Train interactions: {len(train):,}"
)

print("\nKNOWN-ITEM TEST @10")

print(
    f"BPR Recall@10     : "
    f"{bpr_test10['Recall']:.6f}"
)

print(
    f"BPR NDCG@10       : "
    f"{bpr_test10['NDCG']:.6f}"
)

print(
    f"Popularity Recall : "
    f"{pop_test10['Recall']:.6f}"
)

print(
    f"Popularity NDCG   : "
    f"{pop_test10['NDCG']:.6f}"
)

print("\nFULL TEST POPULATION @10")

print(
    f"BPR Recall@10     : "
    f"{full_bpr10['Recall']:.6f}"
)

print(
    f"BPR NDCG@10       : "
    f"{full_bpr10['NDCG']:.6f}"
)

print(
    f"Cold targets      : "
    f"{(~test['product_id'].isin(item_to_idx)).sum():,}"
    f"/{len(test):,}"
)

print("\n✅ STEP 6 COMPLETED")

FINAL STEP 6 AUDIT
✅ selected_model_exists           : True
✅ selected_model_name             : True
✅ train_users_consistent          : True
✅ train_products_consistent       : True
✅ validation_results_exist        : True
✅ test_results_exist              : True
✅ full_test_results_exist         : True
✅ test_timestamp_complete         : True

----------------------------------------------------------------------
STEP 6 FINAL SUMMARY
----------------------------------------------------------------------
Selected CF model : BPR_F64_R05
CF train users    : 26,903
CF train products : 15,546
Train interactions: 205,621

KNOWN-ITEM TEST @10
BPR Recall@10     : 0.004589
BPR NDCG@10       : 0.002375
Popularity Recall : 0.018695
Popularity NDCG   : 0.009259

FULL TEST POPULATION @10
BPR Recall@10     : 0.004535
BPR NDCG@10       : 0.002347
Cold targets      : 318/26,903

✅ STEP 6 COMPLETED
